# Simple GPT Transformer

In [ ]:
from datasets import load_dataset

dataset = load_dataset("amgadmadkour/tiny_shakespeare")
train_dataset = dataset["train"]
print(train_dataset[0])

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset

# --- config ---
block_size, batch_size = 256, 64      # longer context, bigger batch
n_embd, n_head, n_layer = 384, 6, 6   # ~10M params
steps, lr = 5000, 1e-3                # train longer, higher LR
dropout = 0.2
device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available()
          else 'cpu')

# --- data ---
text = "\n".join(load_dataset("amgadmadkour/tiny_shakespeare")["train"]["text"])
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)

# Hold out the last 10% as a validation set so we can measure generalization.
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# --- model ---
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(n_embd, n_head, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        a = self.ln1(x)
        x = x + self.drop(self.attn(a, a, a, attn_mask=mask, need_weights=False)[0])
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)            # final norm before the head (GPT-2 style)
        self.head = nn.Linear(n_embd, vocab_size)
        self.head.weight = self.tok.weight          # weight tying: share input/output embeddings

    def forward(self, idx):
        T = idx.size(1)
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.ln_f(self.blocks(x)))

# --- train ---
@torch.no_grad()
def estimate_loss(iters=50):
    model.eval()
    out = {}
    for split in ('train', 'val'):
        losses = torch.zeros(iters)
        for k in range(iters):
            x, y = get_batch(split)
            losses[k] = F.cross_entropy(model(x).view(-1, vocab_size), y.view(-1))
        out[split] = losses.mean().item()
    model.train()
    return out

print(f"Training on {device}...")
model = GPT().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=lr)
for step in range(steps):
    if step % 300 == 0 or step == steps - 1:
        losses = estimate_loss()
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")
    x, y = get_batch('train')
    loss = F.cross_entropy(model(x).view(-1, vocab_size), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print("Training complete!")

# --- generate (prompt-conditioned completion) ---
@torch.no_grad()
def complete(prompt="", max_new_tokens=500, temperature=1.0, top_k=None):
    model.eval()
    # Encode the prompt into the starting context.
    # Fall back to a single newline/zero token if the prompt is empty.
    if prompt:
        idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    else:
        idx = torch.zeros((1, 1), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # Crop context to the last block_size tokens the model can attend to.
        logits = model(idx[:, -block_size:])
        logits = logits[:, -1, :] / temperature        # focus on the last step

        # Optionally restrict sampling to the top_k most likely tokens.
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_id], dim=1)

    return decode(idx[0].tolist())


# Example: feed a prompt and let the model finish it.
print("Prompted generation:")
print(complete("To be, or not to be", max_new_tokens=500, temperature=0.8, top_k=50))


## Save in Hugging Face-style format

Export the trained custom PyTorch model as `model.safetensors` plus `config.json`, vocab, loader code, and a model card.

In [ ]:
from pathlib import Path
import json
from safetensors.torch import save_file

export_dir = Path("../model/simple-gpt-shakespeare")
export_dir.mkdir(parents=True, exist_ok=True)

config = {
    "model_type": "simple-gpt-char",
    "dataset": "amgadmadkour/tiny_shakespeare",
    "vocab_size": vocab_size,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
}

(export_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n")
(export_dir / "vocab.json").write_text(
    json.dumps({"chars": chars, "stoi": stoi}, indent=2) + "\n"
)

state_dict = {
    name: tensor.detach().cpu().contiguous()
    for name, tensor in model.state_dict().items()
    if name != "head.weight"
}
save_file(state_dict, str(export_dir / "model.safetensors"))

print(f"Saved Hugging Face-style model files to: {export_dir.resolve()}")
print("Files:", sorted(p.name for p in export_dir.iterdir()))


In [ ]:
from huggingface_hub import HfApi

repo_id = "amgadmadkour/simple-gpt-shakespeare"
api = HfApi()
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=export_dir, repo_id=repo_id, repo_type="model")